# Color Picker Experiment

Autonomous color mixing via Bayesian optimization, running cell-by-cell using `ColorPickerNotebook`.

## Setup
- Set `LAB_SERVER_URL` below (or configure via `experiment.settings.yaml` / `EXPERIMENT_LAB_SERVER_URL` env var).
- Each iteration pipettes a batch of colors onto the plate, images it, and feeds the measured RGB values back into the Bayesian solver.
- Run cells sequentially. Re-run individual iteration cells to repeat a step without restarting the experiment.

## Cell 1 — Configuration & Setup

In [4]:
from colorpicker_experiment import ColorPickerNotebook

# ── Lab connection ────────────────────────────────────────────────────────────
LAB_SERVER_URL = "http://parker.cels.anl.gov:8000"  # adjust as needed

# ── Experiment parameters (override config file defaults) ─────────────────────
OPENTRON     = "ot2_gamma"   # OT-2 node name
PIPETTE_SIDE = "left"        # 'left' or 'right'
ITERATIONS   = 4             # number of Bayesian optimization rounds
POP_SIZE     = 4             # colors mixed per iteration

exp = ColorPickerNotebook(
    lab_server_url=LAB_SERVER_URL,
    opentron=OPENTRON,
    pipette_side=PIPETTE_SIDE,
    iterations=ITERATIONS,
    pop_size=POP_SIZE,
)

print(f"Target experiment: {exp.experiment_design.experiment_name}")
print(f"Workcell URL: {getattr(exp, 'workcell_server_url', 'auto-discover from lab')}")

Target experiment: Color Picker
Workcell URL: http://parker.cels.anl.gov:8005/


## Cell 2 — Start Experiment & Initialize Run

In [5]:
exp.start(run_name="Color Picker notebook run")
exp._initialize_run(opentron=OPENTRON, pipette_side=PIPETTE_SIDE)

exp.display(
    {"target_color": exp.target_color, "iterations": ITERATIONS, "pop_size": POP_SIZE},
    title="Run Parameters",
)

╭─────────────────────────────────────────────── Experiment Status ───────────────────────────────────────────────╮
│ Experiment Started                                                                                              │
│ Experiment: Color Picker                                                                                        │
│ ID: 01KJTHT94W25SX7K98ST9GYDDS                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Run Parameters ─────────────────────────────────────────────────╮
│ {'target_color': [155, 87, 78], 'iterations': 4, 'pop_size': 4}                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Cells 3–N — Iteration Loop

Run the cell below once per iteration (change `ITERATION` each time), or duplicate it N times.

In [6]:
ITERATION = 0

colors = exp.loop(ITERATION)
exp.display(colors, title=f"Iteration {ITERATION} — measured well colors")

Mix Colors Workflow['Run OT2 Color Mixing Protocol']: Queued on step 0

.

.

.

Mix Colors Workflow['Run OT2 Color Mixing Protocol']: Running step 0

...

.

.

Mix Colors Workflow['Color Plate from Rotation Deck -> OT2 Deck Pt 2']: Completed Successfully

2026-03-03T19:18:30.611857Z [info     ] EventClient initialized        client_name=madsci.client.data_client event_server=http://parker.cels.anl.gov:8001/ log_dir=/home/rpl/.madsci/logs log_level=EventLogLevel.INFO madsci_version=0.7.0rc1 platform=Linux-6.17.0-14-generic-x86_64-with-glibc2.39 python_version=3.12.3


╭────────────────────────────────────── Iteration 0 — measured well colors ───────────────────────────────────────╮
│ [array([170, 213, 232]), array([170, 214, 235]), array([176, 219, 238]), array([179, 222, 241])]                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
ITERATION = 1

colors = exp.loop(ITERATION)
exp.display(colors, title=f"Iteration {ITERATION} — measured well colors")

In [ ]:
ITERATION = 2

colors = exp.loop(ITERATION)
exp.display(colors, title=f"Iteration {ITERATION} — measured well colors")

In [ ]:
ITERATION = 3

colors = exp.loop(ITERATION)
exp.display(colors, title=f"Iteration {ITERATION} — measured well colors")

## Final Cell — Results & End Experiment

In [ ]:
import numpy as np

best_idx = int(
    np.argmin(
        exp.solver._grade_population(exp.previous_colors, exp.target_color)
    )
)

results = {
    "target_color": exp.target_color,
    "best_color": exp.previous_colors[best_idx],
    "total_wells_used": exp.total_wells,
    "iterations_completed": ITERATIONS,
}

exp.display(results, title="Experiment Results")
exp.end()